# KMeans — Regionais de Saúde de MG por Ano

Unidade de agrupamento: par **(REGIONAL DE SAÚDE, ANO)**. Uma mesma Regional pode cair em agrupamentos diferentes em anos diferentes

O notebook está dividido em duas partes: 
**Parte 1 — Implementação** (entrada, parâmetros, execução do algoritmo e testes de robustez da configuração) 
**Parte 2 — Avaliação e Apresentação** (só resultados: gráficos e caracterização de cada cluster, sem repetir o método).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

sys.path.append(str(Path.cwd()))

from utilitarios_agrupamento_kmeans import (
    COLUNA_ANO, COLUNA_REGIONAL,
    BLOCOS_DE_TAXA, PASTA_TAXA,
    LIMIAR_MAXIMO_DE_NULOS_POR_COLUNA,
    USAR_PCA, RAZAO_MAXIMA_ATRIBUTOS_POR_LINHA, VARIANCIA_EXPLICADA_ALVO,
    NUMERO_MAXIMO_DE_COMPONENTES, SEMENTE_ALEATORIA,
    USAR_JANELA_COMUM, ANO_INICIAL_COMUM, ANO_FINAL_COMUM,
    K_MINIMO, K_MAXIMO, SEMENTES_DA_BUSCA, MINIMO_DE_ELEMENTOS_POR_CLUSTER,
    NUMERO_DE_VARIAVEIS_DISCRIMINANTES,
    montar_painel,
    descartar_colunas_sem_informacao_e_total, descartar_colunas_com_muitos_nulos, tratar_nulos_sem_imputar,
    padronizar_e_reduzir,
    calcular_inercia_e_silhueta_por_k, escolher_k_pelo_cotovelo, busca_em_grade_k,
    maximo_de_iteracoes_usadas,
    executar_kmeans, resumir_agrupamento_kmeans, diagnostico_tamanho_dos_clusters,
    montar_resumo_parametros_kmeans, montar_tabela_pertencimento, montar_visao_agregada_regional_ano,
    montar_resumo_por_cluster,
    calcular_caracterizacao_por_cluster, selecionar_variaveis_mais_discriminantes,
    descrever_caracterizacao_por_cluster,
    calcular_parcelas_distancia_ao_centroide,
    registrar_log,
)
from graficos_agrupamento_kmeans import (
    grafico_cotovelo_inercia,
    grafico_distribuicao_por_regional, grafico_distribuicao_por_ano,
    grafico_heatmap_regional_ano,
    grafico_variaveis_discriminantes_por_cluster,
)

# Parte 1 — Implementação

Entrada dos dados, parâmetros de configuração, execução do algoritmo e os testes que validam a configuração escolhida. Nenhum resultado final é discutido aqui -- isso fica todo na Parte 2.

## 1. Leitura dos CSVs de taxa e reformatação 

In [2]:
painel = montar_painel(blocos=BLOCOS_DE_TAXA, pasta_taxa=PASTA_TAXA)
painel.head()

[LOG] tabela_casos_por_faixa_etária_taxa.csv: 9 categoria(s), 1064 par(es) regional-ano.
[LOG] tabela_casos_por_escolaridade_taxa.csv: 8 categoria(s), 728 par(es) regional-ano.
[LOG] tabela_casos_por_raça_e_cor_taxa.csv: 7 categoria(s), 728 par(es) regional-ano.
[LOG] tabela_mortes_por_faixa_etária_taxa.csv: 9 categoria(s), 1288 par(es) regional-ano.
[LOG] tabela_mamografias_por_faixa_etária_taxa.csv: 9 categoria(s), 364 par(es) regional-ano.
[LOG] Painel montado: 1316 par(es) regional-ano x 42 variável(is), 28 regional(is).


casos-faixa_0-9  casos-faixa_10-19  casos-faixa_20-29  \
REGIONAL DE SAÚDE ANO                                                           
alfenas           1979              NaN                NaN                NaN   
                  1980              NaN                NaN                NaN   
                  1981              NaN                NaN                NaN   
                  1982              NaN                NaN                NaN   
                  1983              NaN                NaN                NaN   

                        casos-faixa_30-39  casos-faixa_40-49  \
REGIONAL DE SAÚDE ANO                                          
alfenas           1979                NaN                NaN   
                  1980                NaN                NaN   
                  1981                NaN                NaN   
                  1982                NaN                NaN   
                  1983                NaN                NaN   

                        casos-faixa_50-59  casos-faixa_60-69  \
REGIONAL DE SAÚDE ANO                                          
alfenas           1979                NaN                NaN   
                  1980                NaN                NaN   
                  1981                NaN                NaN   
                  1982                NaN                NaN   
                  1983                NaN                NaN   

                        casos-faixa_70-ou-mais  casos-faixa_total  \
REGIONAL DE SAÚDE ANO                                               
alfenas           1979                     NaN                NaN   
                  1980                     NaN                NaN   
                  1981                     NaN                NaN   
                  1982                     NaN                NaN   
                  1983                     NaN                NaN   

                        casos-escolaridade_fundamental-completo  ...  \
REGIONAL DE SAÚDE ANO                                            ...   
alfenas           1979                                      NaN  ...   
                  1980                                      NaN  ...   
                  1981                                      NaN  ...   
                  1982                                      NaN  ...   
                  1983                                      NaN  ...   

                        mortalidade_total  mamografia_0-9  mamografia_10-19  \
REGIONAL DE SAÚDE ANO                                                         
alfenas           1979               3.54             NaN               NaN   
                  1980               7.47             NaN               NaN   
                  1981               3.99             NaN               NaN   
                  1982               3.38             NaN               NaN   
                  1983               7.82             NaN               NaN   

                        mamografia_20-29  mamografia_30-39  mamografia_40-49  \
REGIONAL DE SAÚDE ANO                                                          
alfenas           1979               NaN               NaN               NaN   
                  1980               NaN               NaN               NaN   
                  1981               NaN               NaN               NaN   
                  1982               NaN               NaN               NaN   
                  1983               NaN               NaN               NaN   

                        mamografia_50-59  mamografia_60-69  \
REGIONAL DE SAÚDE ANO                                        
alfenas           1979               NaN               NaN   
                  1980               NaN               NaN   
                  1981               NaN               NaN   
                  1982               NaN               NaN   
                  1983               NaN               NaN   

                   

In [3]:
print(
    f"Matriz de entrada: {painel.shape[0]} pares Regional-Ano x {painel.shape[1]} variáveis, "
    f"{painel.index.get_level_values(COLUNA_REGIONAL).nunique()} Regionais de Saúde."
)
pd.set_option("display.max_columns", None)
painel.head(10)

Matriz de entrada: 1316 pares Regional-Ano x 42 variáveis, 28 Regionais de Saúde.


casos-faixa_0-9  casos-faixa_10-19  casos-faixa_20-29  \
REGIONAL DE SAÚDE ANO                                                           
alfenas           1979              NaN                NaN                NaN   
                  1980              NaN                NaN                NaN   
                  1981              NaN                NaN                NaN   
                  1982              NaN                NaN                NaN   
                  1983              NaN                NaN                NaN   
                  1984              NaN                NaN                NaN   
                  1985              NaN                NaN                NaN   
                  1986              NaN                NaN                NaN   
                  1987              NaN                NaN                NaN   
                  1988              NaN                NaN                NaN   

                        casos-faixa_30-39  casos-faixa_40-49  \
REGIONAL DE SAÚDE ANO                                          
alfenas           1979                NaN                NaN   
                  1980                NaN                NaN   
                  1981                NaN                NaN   
                  1982                NaN                NaN   
                  1983                NaN                NaN   
                  1984                NaN                NaN   
                  1985                NaN                NaN   
                  1986                NaN                NaN   
                  1987                NaN                NaN   
                  1988                NaN                NaN   

                        casos-faixa_50-59  casos-faixa_60-69  \
REGIONAL DE SAÚDE ANO                                          
alfenas           1979                NaN                NaN   
                  1980                NaN                NaN   
                  1981                NaN                NaN   
                  1982                NaN                NaN   
                  1983                NaN                NaN   
                  1984                NaN                NaN   
                  1985                NaN                NaN   
                  1986                NaN                NaN   
                  1987                NaN                NaN   
                  1988                NaN                NaN   

                        casos-faixa_70-ou-mais  casos-faixa_total  \
REGIONAL DE SAÚDE ANO                                               
alfenas           1979                     NaN                NaN   
                  1980                     NaN                NaN   
                  1981                     NaN                NaN   
                  1982                     NaN                NaN   
                  1983                     NaN                NaN   
                  1984                     NaN                NaN   
                  1985                     NaN                NaN   
                  1986                     NaN                NaN   
                  1987                     NaN                NaN   
                  1988                     NaN                NaN   

                        casos-escolaridade_fundamental-completo  \
REGIONAL DE SAÚDE ANO                                             
alfenas           1979                                      NaN   
                  1980                                      NaN   
                  1981                                      NaN   
                  1982                                      NaN   
                  1983                                      NaN   
                  1984                                      NaN   
                  1985                                      NaN   
                  1986                                      NaN   
              

In [4]:
matriz_exibicao = descartar_colunas_sem_informacao_e_total(painel).reset_index()
print(
    f"Matriz de entrada (Ano como coluna; sem colunas 'total'/'sem informação'): "
    f"{matriz_exibicao.shape[0]} linhas x {matriz_exibicao.shape[1]} colunas, "
    f"{matriz_exibicao[COLUNA_REGIONAL].nunique()} Regionais de Saúde."
)
matriz_exibicao.head(10)

[LOG] Descartada(s) 7 coluna(s) sem-informação/total: ['casos-faixa_total', 'casos-escolaridade_sem-informacao', 'casos-escolaridade_total', 'casos-raca-cor_sem-informacao', 'casos-raca-cor_total', 'mortalidade_total', 'mamografia_total']
Matriz de entrada (Ano como coluna; sem colunas 'total'/'sem informação'): 1316 linhas x 37 colunas, 28 Regionais de Saúde.


,REGIONAL DE SAÚDE,ANO,casos-faixa_0-9,casos-faixa_10-19,casos-faixa_20-29,casos-faixa_30-39,casos-faixa_40-49,casos-faixa_50-59,casos-faixa_60-69,casos-faixa_70-ou-mais,casos-escolaridade_fundamental-completo,casos-escolaridade_fundamental-incompleto,casos-escolaridade_nenhuma,casos-escolaridade_nivel-medio,casos-escolaridade_nivel-superior-completo,casos-escolaridade_nivel-superior-incompleto,casos-raca-cor_amarela,casos-raca-cor_branca,casos-raca-cor_indigena,casos-raca-cor_parda,casos-raca-cor_preta,mortalidade_0-9,mortalidade_10-19,mortalidade_20-29,mortalidade_30-39,mortalidade_40-49,mortalidade_50-59,mortalidade_60-69,mortalidade_70-ou-mais,mamografia_0-9,mamografia_10-19,mamografia_20-29,mamografia_30-39,mamografia_40-49,mamografia_50-59,mamografia_60-69,mamografia_70-ou-mais
0,alfenas,1979,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.37,28.32,14.14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,alfenas,1980,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.96,21.25,55.53,41.03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,alfenas,1981,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.89,6.94,27.26,13.27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,alfenas,1982,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.93,NaN,17.83,NaN,60.19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,alfenas,1983,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.82,19.98,8.75,50.06,76.91,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,alfenas,1984,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.05,25.76,36.43,110.58,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,alfenas,1985,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.39,42.12,11.78,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,alfenas,1986,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.71,18.78,41.32,11.44,33.85,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,alfenas,1987,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.84,6.13,NaN,33.29,64.86,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,alfenas,1988,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.91,43.07,77.70,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Janela temporal, descarte de colunas e tratamento de nulos


In [5]:
if USAR_JANELA_COMUM:
    anos = painel.index.get_level_values(COLUNA_ANO)
    matriz = painel[(anos >= ANO_INICIAL_COMUM) & (anos <= ANO_FINAL_COMUM)]
else:
    registrar_log("USAR_JANELA_COMUM=False: usando o período completo de cada bloco, sem restringir ano.")
    matriz = painel

matriz = descartar_colunas_sem_informacao_e_total(matriz)
matriz = descartar_colunas_com_muitos_nulos(matriz, limiar_maximo=LIMIAR_MAXIMO_DE_NULOS_POR_COLUNA)
matriz_antes_do_dropna = matriz
matriz = tratar_nulos_sem_imputar(matriz)

registrar_log(f"Matriz final: {matriz.shape[0]} linha(s) x {matriz.shape[1]} atributo(s).")
matriz.head()

[LOG] Descartada(s) 7 coluna(s) sem-informação/total: ['casos-faixa_total', 'casos-escolaridade_sem-informacao', 'casos-escolaridade_total', 'casos-raca-cor_sem-informacao', 'casos-raca-cor_total', 'mortalidade_total', 'mamografia_total']
[LOG] Descartada(s) 6 coluna(s) com mais de 5% de nulos: ['casos-faixa_0-9', 'casos-faixa_10-19', 'mortalidade_0-9', 'mortalidade_10-19', 'mortalidade_20-29', 'mortalidade_30-39']
[LOG] Descartada(s) 24 de 280 linha(s) (8.6%) por ter ao menos um valor ausente -- nenhum valor foi imputado.
[LOG] Matriz final: 256 linha(s) x 29 atributo(s).


casos-faixa_20-29  casos-faixa_30-39  \
REGIONAL DE SAÚDE ANO                                          
alfenas           2014              11.27              86.41   
                  2015              11.37              76.55   
                  2016              14.35              81.29   
                  2017               5.80              44.51   
                  2018               0.00              49.72   

                        casos-faixa_40-49  casos-faixa_50-59  \
REGIONAL DE SAÚDE ANO                                          
alfenas           2014             116.97             160.58   
                  2015             150.01             204.35   
                  2016             118.78             165.55   
                  2017             109.05             138.38   
                  2018             132.57             118.94   

                        casos-faixa_60-69  casos-faixa_70-ou-mais  \
REGIONAL DE SAÚDE ANO                                               
alfenas           2014             241.86                  261.18   
                  2015             177.21                  251.93   
                  2016             194.96                  283.08   
                  2017             159.53                  214.66   
                  2018             140.00                  147.80   

                        casos-escolaridade_fundamental-completo  \
REGIONAL DE SAÚDE ANO                                             
alfenas           2014                                    20.72   
                  2015                                    23.77   
                  2016                                    19.04   
                  2017                                    26.96   
                  2018                                    34.26   

                        casos-escolaridade_fundamental-incompleto  \
REGIONAL DE SAÚDE ANO                                               
alfenas           2014                                      32.23   
                  2015                                      40.22   
                  2016                                      39.90   
                  2017                                      19.32   
                  2018                                      10.23   

                        casos-escolaridade_nenhuma  \
REGIONAL DE SAÚDE ANO                                
alfenas           2014                        4.14   
                  2015                        4.57   
                  2016                        3.17   
                  2017                        2.70   
                  2018                        1.33   

                        casos-escolaridade_nivel-medio  \
REGIONAL DE SAÚDE ANO                                    
alfenas           2014                           12.43   
                  2015                            9.60   
                  2016                           10.88   
                  2017                           12.58   
                  2018                            9.34   

                        casos-escolaridade_nivel-superior-completo  \
REGIONAL DE SAÚDE ANO                                                
alfenas           2014                                       11.51   
                  2015                                       11.43   
                  2016                                       12.69   
                  2017                                        8.09   
                  2018                                       10.23   

                        casos-escolaridade_nivel-superior-incompleto  \
REGIONAL DE SAÚDE ANO                                                  
alfenas           2014                                          0.46   
                  2015                                          1.37   
                  2016                                          0.45   
                  2017                             

### Diagnóstico: o que foi removido, explicitamente

A etapa acima só loga contagens. Aqui vão as identidades: quais colunas caíram (sem-informação/total + esparsas) e, de cada linha (par Regional-Ano) descartada pelo `dropna`, quais colunas específicas estavam nulas nela -- para conferir que o descarte é coerente com a cobertura real dos CSVs de origem, não um bug de junção.

In [6]:
colunas_descartadas = sorted(set(painel.columns) - set(matriz_antes_do_dropna.columns))
print(f"Colunas descartadas (sem-informação/total + >{LIMIAR_MAXIMO_DE_NULOS_POR_COLUNA:.0%} de nulos): {len(colunas_descartadas)} de {painel.shape[1]}")
display(pd.DataFrame({"COLUNA_DESCARTADA": colunas_descartadas}))

linhas_descartadas_idx = matriz_antes_do_dropna.index.difference(matriz.index)
print(f"\nLinhas (pares Regional-Ano) descartadas pelo dropna: {len(linhas_descartadas_idx)} de {len(matriz_antes_do_dropna)}")

mascara_nulos = matriz_antes_do_dropna.loc[linhas_descartadas_idx].isna()
colunas_nulas_por_linha = mascara_nulos.apply(lambda linha: ", ".join(linha.index[linha]), axis=1)
tabela_linhas_descartadas = colunas_nulas_por_linha.reset_index()
tabela_linhas_descartadas.columns = [COLUNA_REGIONAL, COLUNA_ANO, "COLUNAS_COM_NULO"]
tabela_linhas_descartadas

Colunas descartadas (sem-informação/total + >5% de nulos): 13 de 42


,COLUNA_DESCARTADA
0,casos-escolaridade_sem-informacao
1,casos-escolaridade_total
2,casos-faixa_0-9
3,casos-faixa_10-19
4,casos-faixa_total
5,casos-raca-cor_sem-informacao
6,casos-raca-cor_total
7,mamografia_total
8,mortalidade_0-9
9,mortalidade_10-19



Linhas (pares Regional-Ano) descartadas pelo dropna: 24 de 280


,REGIONAL DE SAÚDE,ANO,COLUNAS_COM_NULO
0,diamantina,2017,mortalidade_40-49
1,diamantina,2020,mortalidade_50-59
2,diamantina,2022,mortalidade_40-49
3,ituiutaba,2017,mortalidade_40-49
4,ituiutaba,2018,mortalidade_50-59
5,ituiutaba,2022,mortalidade_40-49
6,ituiutaba,2023,mortalidade_40-49
7,pedra azul,2014,mortalidade_70-ou-mais
8,pedra azul,2015,mortalidade_40-49
9,pedra azul,2018,mortalidade_50-59


### Diagnóstico: proporção de nulos por coluna e cobertura da mamografia em 2013

Proporção de nulos de cada coluna na janela comum, **antes** do descarte por limiar (já sem as colunas de total e sem informação), e a cobertura da base de mamografia em 2013 -- que justifica iniciar a janela em 2014.

In [7]:
anos_painel = painel.index.get_level_values(COLUNA_ANO)
matriz_janela = descartar_colunas_sem_informacao_e_total(
    painel[(anos_painel >= ANO_INICIAL_COMUM) & (anos_painel <= ANO_FINAL_COMUM)]
)
proporcao_de_nulos = matriz_janela.isna().mean().sort_values(ascending=False)
print(f"Proporção de nulos por coluna em {ANO_INICIAL_COMUM}-{ANO_FINAL_COMUM}, antes do descarte (só colunas com algum nulo):")
display(proporcao_de_nulos[proporcao_de_nulos > 0].rename("PROPORCAO_DE_NULOS").to_frame())

colunas_mamografia = []
for coluna in painel.columns:
    if coluna.startswith("mamografia_") and not coluna.endswith("_total"):
        colunas_mamografia.append(coluna)
mamografia_2013 = painel[anos_painel == 2013][colunas_mamografia]
print(
    f"Mamografia em 2013: {mamografia_2013.notna().any(axis=1).sum()} de {len(mamografia_2013)} "
    "Regionais com algum registro. Proporção de nulos por faixa:"
)
mamografia_2013.isna().mean().rename("PROPORCAO_DE_NULOS_2013").to_frame()

[LOG] Descartada(s) 7 coluna(s) sem-informação/total: ['casos-faixa_total', 'casos-escolaridade_sem-informacao', 'casos-escolaridade_total', 'casos-raca-cor_sem-informacao', 'casos-raca-cor_total', 'mortalidade_total', 'mamografia_total']
Proporção de nulos por coluna em 2014-2023, antes do descarte (só colunas com algum nulo):


,PROPORCAO_DE_NULOS
mortalidade_0-9,1.000000
mortalidade_10-19,0.992857
mortalidade_20-29,0.767857
casos-faixa_0-9,0.400000
mortalidade_30-39,0.132143
casos-faixa_10-19,0.100000
mortalidade_40-49,0.042857
mortalidade_50-59,0.021429
mortalidade_70-ou-mais,0.010714
mortalidade_60-69,0.007143


Mamografia em 2013: 3 de 28 Regionais com algum registro. Proporção de nulos por faixa:


,PROPORCAO_DE_NULOS_2013
mamografia_0-9,1.000000
mamografia_10-19,1.000000
mamografia_20-29,1.000000
mamografia_30-39,0.892857
mamografia_40-49,0.892857
mamografia_50-59,0.892857
mamografia_60-69,0.892857
mamografia_70-ou-mais,0.892857


## 3. Padronização (RobustScaler)

Cada variável é centrada na mediana e dividida pela amplitude interquartil (IQR). O KMeans roda diretamente sobre as 29 variáveis padronizadas.

In [8]:
matriz_padronizada, matriz_para_kmeans, pca, variancia_acumulada = padronizar_e_reduzir(
    matriz,
    razao_maxima_atributos_por_linha=RAZAO_MAXIMA_ATRIBUTOS_POR_LINHA,
    variancia_alvo=VARIANCIA_EXPLICADA_ALVO,
    numero_maximo_de_componentes=NUMERO_MAXIMO_DE_COMPONENTES,
    semente=SEMENTE_ALEATORIA,
    usar_pca=USAR_PCA,
)

[LOG] PCA não aplicado: USAR_PCA=False (forçado; razão atributos/linhas = 0.113).


## 4. Escolha de k (cotovelo da curva de inércia)

In [9]:
tabela_k = calcular_inercia_e_silhueta_por_k(matriz_para_kmeans, k_minimo=K_MINIMO, k_maximo=K_MAXIMO, semente=SEMENTE_ALEATORIA)
k_escolhido, indice_do_cotovelo = escolher_k_pelo_cotovelo(tabela_k)

figura_cotovelo = grafico_cotovelo_inercia(tabela_k, indice_do_cotovelo, k_escolhido)
figura_cotovelo.show()
tabela_k.drop(columns="silhueta")

[LOG] Inércia calculada(s) para k = 2..12 (11 valor(es) de k).
[LOG] Cotovelo da curva de inércia na posição 3 de 11 -- k = 5


,k,inercia,n_iter_
0,2,5149.890920,9
1,3,4490.798670,11
2,4,4059.984427,15
3,5,3736.867665,20
4,6,3550.503699,14
5,7,3410.458134,11
6,8,3272.918154,11
7,9,3111.756550,21
8,10,3058.096696,8
9,11,2937.309687,7


### Busca em grade

In [10]:
tabela_da_busca = busca_em_grade_k(matriz_para_kmeans, k_minimo=K_MINIMO, k_maximo=K_MAXIMO, sementes=SEMENTES_DA_BUSCA)
print(f"Resultados da busca em grade: k = {K_MINIMO}..{min(K_MAXIMO, len(matriz_para_kmeans) - 1)}, sementes = {SEMENTES_DA_BUSCA} ({len(tabela_da_busca)} combinações).")

resumo_por_k = tabela_da_busca.groupby("k").agg(
    inercia_media=("inercia", "mean"), inercia_desvio=("inercia", "std"),
).reset_index()
print("\nEstabilidade por k através das sementes testadas:")
resumo_por_k

Resultados da busca em grade: k = 2..12, sementes = [0, 10, 30, 42, 66] (55 combinações).

Estabilidade por k através das sementes testadas:


,k,inercia_media,inercia_desvio
0,2,5149.898274,0.016445
1,3,4492.844804,2.688084
2,4,4062.212502,2.267365
3,5,3737.622877,1.515052
4,6,3560.285626,7.427332
5,7,3404.490254,8.423058
6,8,3250.046507,13.629516
7,9,3126.938436,16.724806
8,10,3038.513994,23.964352
9,11,2947.423474,19.813414


In [11]:
print(f"Variação entre sementes para o k escolhido (k = {k_escolhido}):")
tabela_da_busca[tabela_da_busca["k"] == k_escolhido][["inercia"]].agg(["min", "max"])

Variação entre sementes para o k escolhido (k = 5):


,inercia
min,3736.685750
max,3740.312754


In [12]:
inercia_por_semente = tabela_da_busca.loc[tabela_da_busca["k"] == k_escolhido, "inercia"]
desvio_inercia = inercia_por_semente.std()
media_inercia = inercia_por_semente.mean()
cv_inercia = desvio_inercia / media_inercia * 100

print(f"Coeficiente de variação da inércia entre as {len(inercia_por_semente)} sementes (k = {k_escolhido}):")
print(f"  S  = {desvio_inercia:.4f}")
print(f"  x̄  = {media_inercia:.4f}")
print(f"  CV = {cv_inercia:.4f}%")

Coeficiente de variação da inércia entre as 5 sementes (k = 5):
  S  = 1.5151
  x̄  = 3737.6229
  CV = 0.0405%


In [13]:
maximo_iteracoes = maximo_de_iteracoes_usadas(tabela_k, tabela_da_busca)
max_iter_configurado = KMeans().get_params()["max_iter"]
print(
    f"Máximo de iterações até convergência entre todas as execuções de KMeans "
    f"(cotovelo + busca em grade): {maximo_iteracoes} (max_iter configurado: {max_iter_configurado})."
)


Máximo de iterações até convergência entre todas as execuções de KMeans (cotovelo + busca em grade): 31 (max_iter configurado: 300).


## 5. KMeans

In [14]:
rotulos, modelo_kmeans = executar_kmeans(matriz_para_kmeans, k_escolhido, semente=SEMENTE_ALEATORIA)
n_clusters, inercia_final, silhueta_final = resumir_agrupamento_kmeans(matriz_para_kmeans, rotulos, modelo_kmeans)

registrar_log(f"KMeans(k={k_escolhido}, random_state={SEMENTE_ALEATORIA})")
registrar_log(f"  Agrupamentos: {n_clusters}")
registrar_log(f"  Inércia (WCSS): {inercia_final:.4f}")

_ = diagnostico_tamanho_dos_clusters(rotulos, minimo=MINIMO_DE_ELEMENTOS_POR_CLUSTER)

[LOG] KMeans(k=5, random_state=42)
[LOG]   Agrupamentos: 5
[LOG]   Inércia (WCSS): 3736.8677


## 6. Caracterização dos clusters — como o cálculo é feito

A matriz usada pelo KMeans (`matriz_para_kmeans`) foi padronizada pelo `RobustScaler` -- seus valores não são diretamente interpretáveis (não estão na unidade original de taxa por 100 mil habitantes). Por isso "o que caracteriza cada cluster" é calculado à parte, sobre `matriz` (as variáveis originais, antes de qualquer padronização), cruzada com os rótulos (`rotulos`) que o KMeans devolveu para cada linha. O cálculo, passo a passo (`calcular_caracterizacao_por_cluster`, em `utilitarios_agrupamento_kmeans.py`):

1. **Média geral de cada variável.** Para cada coluna de `matriz` (cada taxa original -- ex. `mortalidade_60-69`), calcula-se a média sobre TODAS as linhas da base, sem agrupar por cluster. Essa é a referência ("perfil médio") contra a qual cada cluster é comparado.
2. **Média da mesma variável, dentro de cada cluster.** As linhas de `matriz` são agrupadas pelo `CLUSTER` de cada par Regional-Ano (`montar_tabela_pertencimento` faz essa junção) e, dentro de cada grupo, calcula-se a média de cada variável -- ou seja, o valor médio daquela taxa só entre os pares que caíram naquele cluster.
3. **Diferença padronizada pelo IQR geral** (Everitt et al., 2011, Seção 5.6.2). Para cada par (cluster, variável):
   ```
   DIFERENCA_PADRONIZADA = (MEDIA_CLUSTER - MEDIA_GERAL) / IQR_GERAL
   ```
   `IQR_GERAL` usa o mesmo Q1/Q3 do `RobustScaler`, então a diferença fica na mesma escala em que o KMeans calculou as distâncias. Positivo = cluster ACIMA da média geral; negativo = ABAIXO. Variáveis com IQR geral 0 ficam `NaN` e vão para o fim da ordenação. A `DIFERENCA_PERCENTUAL` = (MEDIA_CLUSTER - MEDIA_GERAL) / |MEDIA_GERAL| × 100 continua na tabela **só como informativa** -- ela explode quando a média geral é próxima de 0 (categorias raras), por isso não é usada como critério.
4. **Ordenação por relevância.** Dentro de cada cluster, as variáveis são ordenadas pelo valor ABSOLUTO da diferença padronizada, do maior para o menor -- as primeiras linhas são as que mais afastam aquele cluster do perfil médio geral. O sinal (acima/abaixo) não entra no critério de ordenação, só no resultado. Não há teste de significância (ANOVA/Tukey): seria circular, pois o KMeans maximiza a diferença entre grupos nessas mesmas variáveis (Everitt et al., 2011, Seção 9.2, item ix).
5. **Seleção das top-N.** `selecionar_variaveis_mais_discriminantes` pega, de cada cluster já ordenado, só as `NUMERO_DE_VARIAVEIS_DISCRIMINANTES` primeiras linhas -- as variáveis mais discriminantes daquele cluster especificamente (não da base como um todo).
6. **Texto por cluster.** `descrever_caracterizacao_por_cluster` converte essas top-N linhas em uma frase por cluster, no formato "**VARIÁVEL** acima/abaixo da média geral em X IQR (±Y%)", na mesma ordem de relevância -- é a versão em texto corrido da tabela, para leitura direta sem precisar cruzar números manualmente.

Nenhuma etapa interpreta clinicamente a variável (não decide se "mortalidade alta" é bom ou ruim, nem nomeia um "perfil" para o cluster) -- isso é leitura humana, feita na Parte 2 a partir do resultado numérico.

# Parte 2 — Avaliação e Apresentação (Resultados)


## Parâmetros finais e atributos usados

In [15]:
resumo_parametros = montar_resumo_parametros_kmeans(
    k_escolhido, inercia_final, silhueta_final,
    usar_robust_scaler=True, pca=pca, variancia_acumulada=variancia_acumulada,
    n_linhas=matriz.shape[0], n_atributos=matriz.shape[1], n_clusters=n_clusters,
)
resumo_parametros.drop(columns=["PCA_APLICADO", "PCA_N_COMPONENTES", "PCA_VARIANCIA_EXPLICADA", "SILHUETA"])

,K,ESCALONADOR,N_LINHAS,N_ATRIBUTOS,N_CLUSTERS,INERCIA
0,5,RobustScaler,256,29,5,3736.867665


In [16]:
atributos_usados = pd.DataFrame({"ATRIBUTO": list(matriz.columns)})
atributos_usados

,ATRIBUTO
0,casos-faixa_20-29
1,casos-faixa_30-39
2,casos-faixa_40-49
3,casos-faixa_50-59
4,casos-faixa_60-69
5,casos-faixa_70-ou-mais
6,casos-escolaridade_fundamental-completo
7,casos-escolaridade_fundamental-incompleto
8,casos-escolaridade_nenhuma
9,casos-escolaridade_nivel-medio


## Visualizações

In [17]:
resultado = montar_tabela_pertencimento(matriz.index, rotulos)

figura_regional = grafico_distribuicao_por_regional(resultado)
figura_regional.show()

figura_ano = grafico_distribuicao_por_ano(resultado)
figura_ano.show()

### Composição de cada cluster (tabela, não só o gráfico)


In [18]:
resumo_por_cluster = montar_resumo_por_cluster(resultado)
resumo_por_cluster

,CLUSTER,N_PARES,N_REGIONAIS_DISTINTAS,MEMBROS
0,1,41,10,"alfenas (2016), alfenas (2021), alfenas (2022)..."
1,2,60,12,"diamantina (2014), diamantina (2015), diamanti..."
2,3,59,17,"alfenas (2017), alfenas (2018), alfenas (2019)..."
3,4,92,20,"alfenas (2014), alfenas (2015), barbacena (201..."
4,5,4,1,"passos (2019), passos (2021), passos (2022), p..."


## Mapa: cluster por Regional e Ano

In [19]:
resultado

,REGIONAL DE SAÚDE,ANO,CLUSTER
0,alfenas,2014,4
1,alfenas,2015,4
2,alfenas,2016,1
3,alfenas,2017,3
4,alfenas,2018,3
...,...,...,...
251,varginha,2019,4
252,varginha,2020,4
253,varginha,2021,4
254,varginha,2022,4


In [20]:
visao_agregada_regional_ano = montar_visao_agregada_regional_ano(resultado)
visao_agregada_regional_ano

ANO,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
REGIONAL DE SAÚDE,,,,,,,,,,
alfenas,4.0,4.0,1.0,3.0,3.0,3.0,3.0,1.0,1.0,1.0
barbacena,4.0,4.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0
belo horizonte,4.0,4.0,4.0,3.0,3.0,3.0,4.0,4.0,4.0,4.0
coronel fabriciano,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
diamantina,2.0,2.0,2.0,NaN,2.0,2.0,NaN,2.0,NaN,2.0
divinopolis,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
governador valadares,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
itabira,4.0,2.0,3.0,3.0,2.0,2.0,4.0,4.0,1.0,1.0
ituiutaba,2.0,4.0,4.0,NaN,NaN,4.0,2.0,2.0,NaN,NaN


### Heatmap: cluster por Regional e Ano

In [21]:
figura_heatmap = grafico_heatmap_regional_ano(visao_agregada_regional_ano)
figura_heatmap.show()

## Caracterização dos clusters — resultado

O método deste cálculo está na Parte 1, Seção 6. Aqui só o resultado: a tabela com a média de cada variável por cluster (comparada à média geral), o gráfico das variáveis mais discriminantes e, por fim, um resumo em texto corrido do que caracteriza cada cluster.

In [22]:
caracterizacao_por_cluster = calcular_caracterizacao_por_cluster(matriz, rotulos)
variaveis_discriminantes = selecionar_variaveis_mais_discriminantes(
    caracterizacao_por_cluster, top_n=NUMERO_DE_VARIAVEIS_DISCRIMINANTES
)

for cluster in sorted(variaveis_discriminantes["CLUSTER"].unique()):
    print(f"\n=== Cluster {cluster} — variáveis mais discriminantes ===")
    display(
        variaveis_discriminantes[variaveis_discriminantes["CLUSTER"] == cluster]
        .drop(columns="CLUSTER")
        .reset_index(drop=True)
    )

[LOG] ATENÇÃO: 1 variável(is) com IQR geral = 0 (sem diferença padronizada): ['casos-raca-cor_indigena']
[LOG] Caracterização por cluster: 5 cluster(s) x 29 variável(is).

=== Cluster 1 — variáveis mais discriminantes ===


,VARIAVEL,MEDIA_CLUSTER,MEDIA_GERAL,DIFERENCA_PADRONIZADA,DIFERENCA_PERCENTUAL
0,casos-escolaridade_nivel-medio,23.309512,12.836328,1.248666,81.590187
1,casos-raca-cor_preta,10.356585,4.982734,1.237502,107.849437
2,casos-escolaridade_fundamental-completo,17.469024,10.104492,1.051700,72.883744
3,casos-faixa_40-49,140.885122,95.669062,1.046365,47.262990
4,casos-faixa_50-59,197.936585,138.746680,1.020868,42.660412
5,casos-escolaridade_nivel-superior-completo,14.320976,8.083867,0.985714,77.155009
6,casos-faixa_60-69,231.117805,162.991250,0.984061,41.797676
7,casos-raca-cor_branca,58.578293,29.861523,0.921739,96.166458
8,casos-escolaridade_fundamental-incompleto,33.780976,21.388008,0.857793,57.943535
9,casos-faixa_70-ou-mais,240.420488,163.081602,0.822076,47.423428



=== Cluster 2 — variáveis mais discriminantes ===


,VARIAVEL,MEDIA_CLUSTER,MEDIA_GERAL,DIFERENCA_PADRONIZADA,DIFERENCA_PERCENTUAL
0,casos-faixa_60-69,90.800667,162.991250,-1.042764,-44.291079
1,casos-faixa_70-ou-mais,80.056167,163.081602,-0.882522,-50.910363
2,casos-faixa_50-59,88.547833,138.746680,-0.865796,-36.180215
3,casos-escolaridade_fundamental-completo,4.873833,10.104492,-0.746970,-51.765678
4,casos-faixa_40-49,63.977500,95.669062,-0.733389,-33.126239
5,casos-escolaridade_nivel-medio,6.719500,12.836328,-0.729279,-47.652476
6,casos-escolaridade_nivel-superior-incompleto,0.314833,1.064453,-0.673816,-70.422997
7,mortalidade_70-ou-mais,39.159500,62.208164,-0.673691,-37.050867
8,casos-escolaridade_fundamental-incompleto,11.739167,21.388008,-0.667855,-45.113323
9,casos-raca-cor_branca,9.426833,29.861523,-0.655904,-68.431506



=== Cluster 3 — variáveis mais discriminantes ===


,VARIAVEL,MEDIA_CLUSTER,MEDIA_GERAL,DIFERENCA_PADRONIZADA,DIFERENCA_PERCENTUAL
0,mamografia_30-39,1069.826780,566.908203,1.214288,88.712524
1,mamografia_70-ou-mais,3686.771864,1958.028047,1.084143,88.290044
2,mamografia_40-49,8210.709492,4721.342070,1.029645,73.906262
3,mamografia_20-29,94.004746,52.386641,0.807335,79.444119
4,mamografia_10-19,12.115763,6.282383,0.734451,92.852984
5,mamografia_60-69,15340.573051,10546.697773,0.693047,45.453803
6,mamografia_50-59,18189.416610,12650.522656,0.663941,43.783914
7,mamografia_0-9,8.024407,5.404258,0.340943,48.483049
8,casos-faixa_20-29,3.571525,5.372773,-0.334494,-33.525479
9,mortalidade_70-ou-mais,71.912373,62.208164,0.283645,15.599574



=== Cluster 4 — variáveis mais discriminantes ===


,VARIAVEL,MEDIA_CLUSTER,MEDIA_GERAL,DIFERENCA_PADRONIZADA,DIFERENCA_PERCENTUAL
0,mamografia_40-49,2861.826522,4721.342070,-0.548707,-39.385317
1,mamografia_30-39,360.213587,566.908203,-0.499060,-36.459980
2,mamografia_20-29,28.977065,52.386641,-0.454114,-44.686155
3,mamografia_50-59,8868.576739,12650.522656,-0.453337,-29.895570
4,mamografia_60-69,7490.352065,10546.697773,-0.441854,-28.979172
5,mamografia_70-ou-mais,1292.373478,1958.028047,-0.417450,-33.996171
6,mamografia_10-19,3.790326,6.282383,-0.313762,-39.667381
7,casos-escolaridade_nenhuma,3.728478,3.012734,0.280409,23.757285
8,mamografia_0-9,3.570435,5.404258,-0.238624,-33.932930
9,casos-escolaridade_nivel-superior-incompleto,0.813152,1.064453,-0.225888,-23.608456



=== Cluster 5 — variáveis mais discriminantes ===


,VARIAVEL,MEDIA_CLUSTER,MEDIA_GERAL,DIFERENCA_PADRONIZADA,DIFERENCA_PERCENTUAL
0,casos-escolaridade_nivel-superior-incompleto,14.0825,1.064453,11.701615,1222.979817
1,mamografia_40-49,21030.8825,4721.342070,4.812632,345.442889
2,mamografia_20-29,218.6475,52.386641,3.225235,317.372630
3,mamografia_10-19,23.8750,6.282383,2.214997,280.030965
4,mamografia_30-39,1410.3775,566.908203,2.036541,148.784105
5,casos-escolaridade_nivel-medio,28.3500,12.836328,1.849618,120.857552
6,casos-escolaridade_fundamental-completo,0.1200,10.104492,-1.425847,-98.812409
7,mamografia_70-ou-mais,3846.2250,1958.028047,1.184140,96.433601
8,mamografia_50-59,21814.2950,12650.522656,1.098450,72.437895
9,mamografia_60-69,17805.2200,10546.697773,1.049360,68.822701


In [23]:
figura_discriminantes = grafico_variaveis_discriminantes_por_cluster(variaveis_discriminantes)
figura_discriminantes.show()

### Resumo em texto: o que caracteriza cada cluster

Mesma informação da tabela e do gráfico acima, em texto corrido -- as variáveis mais discriminantes de cada cluster, na ordem de relevância, com o sinal, a diferença em IQR e o percentual (informativo) em relação à média geral da base.

In [24]:
from IPython.display import Markdown

display(Markdown(descrever_caracterizacao_por_cluster(caracterizacao_por_cluster, resumo_por_cluster)))

- **Cluster 1** (41 pares Regional-Ano): **casos-escolaridade_nivel-medio** acima da média geral em 1.25 IQR (+81.6%); **casos-raca-cor_preta** acima da média geral em 1.24 IQR (+107.8%); **casos-escolaridade_fundamental-completo** acima da média geral em 1.05 IQR (+72.9%); **casos-faixa_40-49** acima da média geral em 1.05 IQR (+47.3%); **casos-faixa_50-59** acima da média geral em 1.02 IQR (+42.7%); **casos-escolaridade_nivel-superior-completo** acima da média geral em 0.99 IQR (+77.2%); **casos-faixa_60-69** acima da média geral em 0.98 IQR (+41.8%); **casos-raca-cor_branca** acima da média geral em 0.92 IQR (+96.2%); **casos-escolaridade_fundamental-incompleto** acima da média geral em 0.86 IQR (+57.9%); **casos-faixa_70-ou-mais** acima da média geral em 0.82 IQR (+47.4%); **casos-faixa_30-39** acima da média geral em 0.65 IQR (+40.6%); **casos-faixa_20-29** acima da média geral em 0.61 IQR (+60.7%); **casos-escolaridade_nivel-superior-incompleto** acima da média geral em 0.53 IQR (+55.8%); **mamografia_0-9** acima da média geral em 0.49 IQR (+69.9%); **casos-raca-cor_amarela** acima da média geral em 0.43 IQR (+58.7%).
- **Cluster 2** (60 pares Regional-Ano): **casos-faixa_60-69** abaixo da média geral em 1.04 IQR (-44.3%); **casos-faixa_70-ou-mais** abaixo da média geral em 0.88 IQR (-50.9%); **casos-faixa_50-59** abaixo da média geral em 0.87 IQR (-36.2%); **casos-escolaridade_fundamental-completo** abaixo da média geral em 0.75 IQR (-51.8%); **casos-faixa_40-49** abaixo da média geral em 0.73 IQR (-33.1%); **casos-escolaridade_nivel-medio** abaixo da média geral em 0.73 IQR (-47.7%); **casos-escolaridade_nivel-superior-incompleto** abaixo da média geral em 0.67 IQR (-70.4%); **mortalidade_70-ou-mais** abaixo da média geral em 0.67 IQR (-37.1%); **casos-escolaridade_fundamental-incompleto** abaixo da média geral em 0.67 IQR (-45.1%); **casos-raca-cor_branca** abaixo da média geral em 0.66 IQR (-68.4%); **casos-escolaridade_nivel-superior-completo** abaixo da média geral em 0.65 IQR (-50.9%); **casos-raca-cor_preta** abaixo da média geral em 0.63 IQR (-55.2%); **casos-faixa_30-39** abaixo da média geral em 0.63 IQR (-39.3%); **mamografia_70-ou-mais** abaixo da média geral em 0.56 IQR (-46.0%); **mamografia_30-39** abaixo da média geral em 0.55 IQR (-40.3%).
- **Cluster 3** (59 pares Regional-Ano): **mamografia_30-39** acima da média geral em 1.21 IQR (+88.7%); **mamografia_70-ou-mais** acima da média geral em 1.08 IQR (+88.3%); **mamografia_40-49** acima da média geral em 1.03 IQR (+73.9%); **mamografia_20-29** acima da média geral em 0.81 IQR (+79.4%); **mamografia_10-19** acima da média geral em 0.73 IQR (+92.9%); **mamografia_60-69** acima da média geral em 0.69 IQR (+45.5%); **mamografia_50-59** acima da média geral em 0.66 IQR (+43.8%); **mamografia_0-9** acima da média geral em 0.34 IQR (+48.5%); **casos-faixa_20-29** abaixo da média geral em 0.33 IQR (-33.5%); **mortalidade_70-ou-mais** acima da média geral em 0.28 IQR (+15.6%); **mortalidade_60-69** acima da média geral em 0.14 IQR (+8.2%); **casos-escolaridade_nivel-superior-incompleto** abaixo da média geral em 0.13 IQR (-13.3%); **casos-faixa_40-49** abaixo da média geral em 0.12 IQR (-5.5%); **casos-raca-cor_parda** abaixo da média geral em 0.11 IQR (-9.3%); **casos-raca-cor_preta** abaixo da média geral em 0.11 IQR (-9.7%).
- **Cluster 4** (92 pares Regional-Ano): **mamografia_40-49** abaixo da média geral em 0.55 IQR (-39.4%); **mamografia_30-39** abaixo da média geral em 0.50 IQR (-36.5%); **mamografia_20-29** abaixo da média geral em 0.45 IQR (-44.7%); **mamografia_50-59** abaixo da média geral em 0.45 IQR (-29.9%); **mamografia_60-69** abaixo da média geral em 0.44 IQR (-29.0%); **mamografia_70-ou-mais** abaixo da média geral em 0.42 IQR (-34.0%); **mamografia_10-19** abaixo da média geral em 0.31 IQR (-39.7%); **casos-escolaridade_nenhuma** acima da média geral em 0.28 IQR (+23.8%); **mamografia_0-9** abaixo da média geral em 0.24 IQR (-33.9%); **casos-escolaridade_nivel-superior-incompleto** abaixo da média geral em 0.23 IQR (-23.6%); **casos-faixa_60-69** acima da média geral em 0.21 IQR (+9.0%); **mortalidade_70-ou-mais** acima da média geral em 0.21 IQR (+11.4%); **casos-escolaridade_nivel-medio** abaixo da média geral em 0.18 IQR (-11.7%); **mortalidade_50-59** acima da média geral em 0.17 IQR (+10.1%); **casos-faixa_70-ou-mais** acima da média geral em 0.17 IQR (+9.7%).
- **Cluster 5** (4 pares Regional-Ano): **casos-escolaridade_nivel-superior-incompleto** acima da média geral em 11.70 IQR (+1223.0%); **mamografia_40-49** acima da média geral em 4.81 IQR (+345.4%); **mamografia_20-29** acima da média geral em 3.23 IQR (+317.4%); **mamografia_10-19** acima da média geral em 2.21 IQR (+280.0%); **mamografia_30-39** acima da média geral em 2.04 IQR (+148.8%); **casos-escolaridade_nivel-medio** acima da média geral em 1.85 IQR (+120.9%); **casos-escolaridade_fundamental-completo** abaixo da média geral em 1.43 IQR (-98.8%); **mamografia_70-ou-mais** acima da média geral em 1.18 IQR (+96.4%); **mamografia_50-59** acima da média geral em 1.10 IQR (+72.4%); **mamografia_60-69** acima da média geral em 1.05 IQR (+68.8%); **mamografia_0-9** acima da média geral em 0.92 IQR (+130.7%); **casos-faixa_20-29** acima da média geral em 0.71 IQR (+71.3%); **casos-faixa_40-49** acima da média geral em 0.62 IQR (+27.8%); **casos-escolaridade_fundamental-incompleto** acima da média geral em 0.59 IQR (+39.7%); **casos-faixa_50-59** acima da média geral em 0.58 IQR (+24.3%).

## Caracterização individual — decomposição da distância ao centroide

A caracterização por médias (acima) descreve cada cluster como um todo. Aqui, a distância² de cada par Regional-Ano até o centroide do seu cluster -- a mesma que o KMeans usa -- é decomposta em uma parcela por variável, `(valor padronizado − centroide)²` (`calcular_parcelas_distancia_ao_centroide`). A soma das parcelas de um par é a distância² dele; a soma de todas é a inércia final.

In [25]:
parcelas_distancia = calcular_parcelas_distancia_ao_centroide(matriz_para_kmeans, modelo_kmeans, matriz.index, matriz.columns)
assert np.isclose(parcelas_distancia["PARCELA_DISTANCIA"].sum(), inercia_final), "soma das parcelas != inércia final"
print(f"Soma de todas as parcelas = {parcelas_distancia['PARCELA_DISTANCIA'].sum():.4f} | inércia final = {inercia_final:.4f}")

distancia_por_par = (
    parcelas_distancia.groupby([COLUNA_REGIONAL, COLUNA_ANO, "CLUSTER"])["PARCELA_DISTANCIA"].sum()
    .rename("DISTANCIA2").sort_values(ascending=False).reset_index()
)
print("\nPares mais distantes do centroide do seu cluster:")
distancia_por_par.head(10)

Soma de todas as parcelas = 3736.8677 | inércia final = 3736.8677

Pares mais distantes do centroide do seu cluster:


,REGIONAL DE SAÚDE,ANO,CLUSTER,DISTANCIA2
0,passos,2017,3,105.663292
1,sete lagoas,2022,1,65.904306
2,leopoldina,2019,1,64.811341
3,alfenas,2018,3,63.525769
4,patos de minas,2023,1,62.651679
5,passos,2018,3,61.472790
6,coronel fabriciano,2018,4,60.155827
7,alfenas,2023,1,47.173062
8,alfenas,2022,1,38.520216
9,passos,2019,5,37.974725


### Variáveis que mais pesam na distância dos pares mais distantes

Para os quatro pares mais distantes: as três maiores parcelas, a participação de cada uma na distância² do par e o valor original da taxa, comparado à média e à mediana gerais.

In [26]:
for _, par in distancia_por_par.head(4).iterrows():
    regional, ano = par[COLUNA_REGIONAL], par[COLUNA_ANO]
    maiores = parcelas_distancia[
        (parcelas_distancia[COLUNA_REGIONAL] == regional) & (parcelas_distancia[COLUNA_ANO] == ano)
    ].head(3).copy()
    maiores["PARTICIPACAO_NO_PAR"] = maiores["PARCELA_DISTANCIA"] / par["DISTANCIA2"]
    maiores["VALOR_ORIGINAL"] = maiores["VARIAVEL"].map(matriz.loc[(regional, ano)])
    maiores["MEDIA_GERAL"] = maiores["VARIAVEL"].map(matriz.mean())
    maiores["MEDIANA_GERAL"] = maiores["VARIAVEL"].map(matriz.median())
    print(f"\n{regional} {ano} -- Cluster {par['CLUSTER']}, distância² = {par['DISTANCIA2']:.2f}")
    display(maiores.drop(columns=[COLUNA_REGIONAL, COLUNA_ANO, "CLUSTER"]).reset_index(drop=True))


passos 2017 -- Cluster 3, distância² = 105.66


,VARIAVEL,PARCELA_DISTANCIA,PARTICIPACAO_NO_PAR,VALOR_ORIGINAL,MEDIA_GERAL,MEDIANA_GERAL
0,mamografia_40-49,20.097846,0.190207,23403.37,4721.342070,3859.670
1,mamografia_70-ou-mais,15.999589,0.151421,10064.98,1958.028047,1620.410
2,mamografia_10-19,15.366042,0.145425,43.25,6.282383,4.455



sete lagoas 2022 -- Cluster 1, distância² = 65.90


,VARIAVEL,PARCELA_DISTANCIA,PARTICIPACAO_NO_PAR,VALOR_ORIGINAL,MEDIA_GERAL,MEDIANA_GERAL
0,casos-raca-cor_amarela,42.123266,0.639158,3.83,0.368437,0.000
1,casos-raca-cor_parda,8.841152,0.134151,91.70,25.235977,21.895
2,casos-raca-cor_preta,2.258876,0.034275,3.83,4.982734,3.990



leopoldina 2019 -- Cluster 1, distância² = 64.81


,VARIAVEL,PARCELA_DISTANCIA,PARTICIPACAO_NO_PAR,VALOR_ORIGINAL,MEDIA_GERAL,MEDIANA_GERAL
0,mamografia_0-9,16.427851,0.253472,40.33,5.404258,3.050
1,casos-escolaridade_nenhuma,11.982338,0.184880,12.36,3.012734,2.405
2,casos-raca-cor_preta,10.940464,0.168805,24.72,4.982734,3.990



alfenas 2018 -- Cluster 3, distância² = 63.53


,VARIAVEL,PARCELA_DISTANCIA,PARTICIPACAO_NO_PAR,VALOR_ORIGINAL,MEDIA_GERAL,MEDIANA_GERAL
0,casos-raca-cor_amarela,41.880641,0.659270,3.56,0.368437,0.00
1,casos-escolaridade_fundamental-completo,12.532591,0.197284,34.26,10.104492,7.99
2,mortalidade_50-59,1.126225,0.017729,10.20,26.984414,25.86


### Participação de cada variável na inércia total e homogeneidade dos clusters

In [27]:
participacao_na_inercia = (
    parcelas_distancia.groupby("VARIAVEL")["PARCELA_DISTANCIA"].sum() / inercia_final
).sort_values(ascending=False).rename("PARTICIPACAO_NA_INERCIA")
print("Variáveis que mais contribuem para a inércia total:")
display(participacao_na_inercia.head(10).to_frame())

print("Distância² ao centroide por cluster (menor média = cluster mais homogêneo):")
distancia_por_par.groupby("CLUSTER")["DISTANCIA2"].agg(["mean", "median", "max"])

Variáveis que mais contribuem para a inércia total:


,PARTICIPACAO_NA_INERCIA
VARIAVEL,
casos-raca-cor_amarela,0.086171
casos-escolaridade_fundamental-completo,0.064650
casos-escolaridade_nenhuma,0.062349
mamografia_0-9,0.051723
casos-escolaridade_nivel-superior-incompleto,0.050893
casos-faixa_20-29,0.044873
casos-raca-cor_parda,0.043444
mortalidade_40-49,0.037521
casos-raca-cor_preta,0.036278


Distância² ao centroide por cluster (menor média = cluster mais homogêneo):


,mean,median,max
CLUSTER,,,
1,22.421182,16.521829,65.904306
2,9.044987,8.074936,22.300850
3,17.560372,13.116960,105.663292
4,12.581002,10.748480,60.155827
5,20.346469,15.104193,37.974725


### Outros valores citados no texto da monografia

Categorias de escolaridade do Cluster 1 (todas, não só as do top-10) e a trajetória de Passos nas variáveis que definem o Cluster 5.

In [28]:
escolaridade_cluster_1 = caracterizacao_por_cluster[
    (caracterizacao_por_cluster["CLUSTER"] == 1)
    & caracterizacao_por_cluster["VARIAVEL"].str.startswith("casos-escolaridade")
]
display(escolaridade_cluster_1.reset_index(drop=True))

matriz.xs("passos", level=COLUNA_REGIONAL)[["casos-escolaridade_nivel-superior-incompleto", "mamografia_40-49"]]

,CLUSTER,VARIAVEL,MEDIA_CLUSTER,MEDIA_GERAL,DIFERENCA_PADRONIZADA,DIFERENCA_PERCENTUAL
0,1,casos-escolaridade_nivel-medio,23.309512,12.836328,1.248666,81.590187
1,1,casos-escolaridade_fundamental-completo,17.469024,10.104492,1.051700,72.883744
2,1,casos-escolaridade_nivel-superior-completo,14.320976,8.083867,0.985714,77.155009
3,1,casos-escolaridade_fundamental-incompleto,33.780976,21.388008,0.857793,57.943535
4,1,casos-escolaridade_nivel-superior-incompleto,1.658537,1.064453,0.534008,55.811143
5,1,casos-escolaridade_nenhuma,3.524390,3.012734,0.200453,16.983106


,casos-escolaridade_nivel-superior-incompleto,mamografia_40-49
ANO,,
2014,0.00,1228.91
2015,0.00,3735.71
2016,2.08,14825.19
2017,0.00,23403.37
2018,5.10,16816.91
2019,8.56,16786.69
2020,3.48,9399.47
2021,13.75,21074.69
2022,17.31,21956.46
